# RAG - Retrieval Augmented Generation

Чат с документами: LLM отвечает на основе твоих данных.

**Стек:**
- ChromaDB — векторная база данных
- Sentence Transformers — embeddings (бесплатно, локально)
- BM25 — keyword search
- DeepSeek — генерация ответов

In [1]:
# Загрузка зависимостей
from dotenv import load_dotenv
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from rank_bm25 import BM25Okapi
from openai import OpenAI

load_dotenv("../../.env")
print("Зависимости загружены")

C:\Users\Ruslan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Зависимости загружены


## 1. Документы

In [2]:
# Тестовые документы
documents = [
    {"id": "1", "text": "Компания ТехноСофт основана в 2015 году в Москве. Основатель - Иван Петров. Компания специализируется на разработке корпоративного ПО и AI-решений.", "source": "about.txt"},
    {"id": "2", "text": "В ТехноСофт работает 150 сотрудников. Офисы расположены в Москве и Санкт-Петербурге. Средняя зарплата разработчика - 250 000 рублей.", "source": "company.txt"},
    {"id": "3", "text": "Основные продукты ТехноСофт: CRM-система 'Клиент+', ERP-система 'Управление', AI-платформа 'Нейро'. CRM используют более 500 компаний.", "source": "products.txt"},
    {"id": "4", "text": "Отпуск в ТехноСофт составляет 28 календарных дней. Также предоставляется ДМС, бесплатные обеды и компенсация спортзала до 5000 рублей в месяц.", "source": "benefits.txt"},
    {"id": "5", "text": "Технический стек ТехноСофт: Java, Spring Boot, PostgreSQL, React, Python, TensorFlow. Для CI/CD используется GitLab и Kubernetes.", "source": "tech.txt"},
    {"id": "6", "text": "Контакты ТехноСофт: email - info@technosoft.ru, телефон - +7 (495) 123-45-67. Адрес главного офиса: Москва, ул. Программистов, д. 10.", "source": "contacts.txt"},
    {"id": "7", "text": "Генеральный директор ТехноСофт - Мария Сидорова. Технический директор - Алексей Козлов. HR-директор - Елена Новикова.", "source": "team.txt"},
    {"id": "8", "text": "Выручка ТехноСофт в 2025 году составила 500 млн рублей. Рост по сравнению с 2024 годом - 40%. Компания планирует IPO в 2027 году.", "source": "finance.txt"}
]

print(f"Загружено {len(documents)} документов")

Загружено 8 документов


## 2. Embeddings + ChromaDB

In [3]:
# Модель для embeddings
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print(f"Embedding модель загружена, размерность: {embedding_model.get_sentence_embedding_dimension()}")

# ChromaDB
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="technosoft_docs")

# Добавляем документы
texts = [doc["text"] for doc in documents]
embeddings = embedding_model.encode(texts).tolist()

collection.add(
    ids=[doc["id"] for doc in documents],
    documents=texts,
    embeddings=embeddings,
    metadatas=[{"source": doc["source"]} for doc in documents]
)

print(f"ChromaDB: {collection.count()} документов")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 545.89it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding модель загружена, размерность: 384
ChromaDB: 8 документов


## 3. BM25 (Keyword Search)

In [4]:
# BM25 для поиска по ключевым словам
tokenized_docs = [doc["text"].lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

print(f"BM25 индекс создан для {len(tokenized_docs)} документов")

BM25 индекс создан для 8 документов


## 4. Гибридный поиск (Vector + BM25)

In [5]:
def hybrid_search(query: str, n_results: int = 3, alpha: float = 0.5):
    """
    Гибридный поиск: BM25 + Vector.
    alpha: 0 = только BM25, 1 = только vector
    """
    # BM25 scores
    bm25_scores = bm25.get_scores(query.lower().split())
    if bm25_scores.max() > 0:
        bm25_scores = bm25_scores / bm25_scores.max()
    
    # Vector scores
    query_emb = embedding_model.encode(query).tolist()
    vec_results = collection.query(query_embeddings=[query_emb], n_results=len(documents))
    
    # Собираем hybrid scores
    results = []
    for i, doc_id in enumerate(vec_results['ids'][0]):
        idx = int(doc_id) - 1
        dist = vec_results['distances'][0][i]
        vec_score = 1 / (1 + dist)
        hybrid = alpha * vec_score + (1 - alpha) * bm25_scores[idx]
        
        results.append({
            "idx": idx,
            "text": documents[idx]["text"],
            "source": documents[idx]["source"],
            "bm25": round(bm25_scores[idx], 3),
            "vector": round(vec_score, 3),
            "hybrid": round(hybrid, 3)
        })
    
    results.sort(key=lambda x: x["hybrid"], reverse=True)
    return results[:n_results]

print("Функция hybrid_search готова")

Функция hybrid_search готова


In [6]:
# Тест: сравнение методов поиска
test_queries = ["Какая выручка?", "Кто основатель?", "Какая зарплата?"]

for q in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {q}")
    print(f"{'Источник':<15} {'BM25':<8} {'Vector':<8} {'Hybrid':<8}")
    print("-"*40)
    for r in hybrid_search(q, n_results=3):
        print(f"{r['source']:<15} {r['bm25']:<8} {r['vector']:<8} {r['hybrid']:<8}")


Query: Какая выручка?
Источник        BM25     Vector   Hybrid  
----------------------------------------
benefits.txt    0.0      0.093    0.046   
contacts.txt    0.0      0.079    0.039   
tech.txt        0.0      0.068    0.034   

Query: Кто основатель?
Источник        BM25     Vector   Hybrid  
----------------------------------------
benefits.txt    0.0      0.093    0.046   
contacts.txt    0.0      0.079    0.039   
tech.txt        0.0      0.068    0.034   

Query: Какая зарплата?
Источник        BM25     Vector   Hybrid  
----------------------------------------
benefits.txt    0.0      0.093    0.046   
contacts.txt    0.0      0.079    0.039   
tech.txt        0.0      0.068    0.034   


## 5. RAG: Поиск + LLM

In [7]:
# DeepSeek клиент
llm = OpenAI(api_key=os.getenv('DEEPSEEK_API_KEY'), base_url="https://api.deepseek.com")

def rag_answer(question: str, n_docs: int = 3) -> str:
    """RAG: поиск документов + генерация ответа."""
    # 1. Поиск
    docs = hybrid_search(question, n_results=n_docs)
    context = "\n\n".join([f"[{d['source']}]: {d['text']}" for d in docs])
    
    # 2. Генерация
    response = llm.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": "Отвечай ТОЛЬКО на основе контекста. Если нет информации - скажи что не знаешь. Кратко, на русском."},
            {"role": "user", "content": f"Контекст:\n{context}\n\nВопрос: {question}"}
        ],
        max_tokens=300,
        temperature=0.3
    )
    return response.choices[0].message.content

print("RAG готов")

RAG готов


In [8]:
# Тестируем RAG
questions = [
    "Какая выручка компании?",
    "Кто основатель и когда создана?",
    "Какая зарплата и бонусы?",
    "Какие продукты выпускает?",
    "Какой технический стек?"
]

for q in questions:
    print(f"\nQ: {q}")
    print(f"A: {rag_answer(q)}")
    print("-"*50)


Q: Какая выручка компании?
A: Выручка компании ТехноСофт в 2025 году составила 500 млн рублей.
--------------------------------------------------

Q: Кто основатель и когда создана?
A: Основатель — Иван Петров, компания создана в 2015 году.
--------------------------------------------------

Q: Какая зарплата и бонусы?
A: На основе контекста: средняя зарплата разработчика — 250 000 рублей. Бонусы включают ДМС, бесплатные обеды и компенсацию спортзала до 5000 рублей в месяц.
--------------------------------------------------

Q: Какие продукты выпускает?
A: Основные продукты ТехноСофт: CRM-система 'Клиент+', ERP-система 'Управление', AI-платформа 'Нейро'.
--------------------------------------------------

Q: Какой технический стек?
A: На основе контекста технический стек ТехноСофт включает: Java, Spring Boot, PostgreSQL, React, Python, TensorFlow. Для CI/CD используется GitLab и Kubernetes.
--------------------------------------------------


## 6. Добавление документов

In [9]:
def add_document(doc_id: str, text: str, source: str):
    """Добавить документ в обе базы (ChromaDB + BM25)."""
    global bm25, tokenized_docs
    
    # ChromaDB
    embedding = embedding_model.encode(text).tolist()
    collection.add(ids=[doc_id], documents=[text], embeddings=[embedding], metadatas=[{"source": source}])
    
    # BM25 + documents list
    documents.append({"id": doc_id, "text": text, "source": source})
    tokenized_docs.append(text.lower().split())
    bm25 = BM25Okapi(tokenized_docs)
    
    print(f"Добавлен: {source} (всего: {len(documents)})")

# Добавляем новый документ
add_document("9", "В ТехноСофт есть стажировка для студентов. Длительность 3 месяца, оплата 50 000 руб/мес.", "internship.txt")

Добавлен: internship.txt (всего: 9)


In [10]:
# Проверяем новый документ
print(rag_answer("Есть ли стажировка?"))

Да, в ТехноСофт есть стажировка для студентов.


## Итоги

**RAG пайплайн:**
1. Документы → Embeddings → ChromaDB (vector search)
2. Документы → Токенизация → BM25 (keyword search)
3. Query → Hybrid Search (vector + BM25)
4. Top-K документов → LLM → Ответ

**Проблема:** Embedding модель слабая для русского → BM25 спасает

**Улучшения для продакшна:**
- Лучшая модель: `intfloat/multilingual-e5-large` или OpenAI embeddings
- Chunking больших документов
- Reranking (Cohere, Cross-encoder)
- Персистентный ChromaDB